System construction and test


In [2]:
from datetime import date, datetime
import pandas as pd
import time
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'
from itertools import product
import sqlite3
from Stoch_HighMedLow_Long import *
from stoploss import *
from index import *
from metrics import *
from backtest import *
from concurrent.futures import ThreadPoolExecutor
import traceback

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)


In [5]:
#seleçonar backtest :  1	1	2020-04-03 19:30:00	2024-04-03 19:30:00	GGAL	60min	USD	C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db
# no DB sqTradeSys

id_backtest = 1

In [7]:
import sqlite3
import pandas as pd

            #conecta ao banco sqTradeSys  e exetrae os parametros gerales do back test da vista vwbacktest 
def backtest_parameters (id_backtest) :   
    
    # Conecta ao banco de dados
    con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite
    
    # Consulta SQL para extrair o registro
    query = f"SELECT * FROM vwbacktest WHERE id_backtest = {id_backtest}"
    
    # Executa a consulta e lê em um DataFrame
    df = pd.read_sql_query(query, con)
    
    # Converte o primeiro (e único) registro em Series
    srbacktest = df.iloc[0] if not df.empty else None
    
    # Fecha a conexão (opcional)
    con.close()
    return srbacktest

srbacktest = backtest_parameters (id_backtest)
display (srbacktest)

id_backtest                                                    1
id_titulos                                                     1
dataini                                      2020-04-03 19:30:00
datafim                                      2024-04-03 19:30:00
symbol                                                      GGAL
intervalo                                                  60min
moeda                                                        USD
source         C:\Users\scitr\anaconda_projects\Trading_Syste...
Name: 0, dtype: object

In [9]:
import sqlite3
import pandas as pd

                                #importa os dados historicos do Titulo a testar

def titulos_dados (srbacktest) :
    # Caminho para o banco de dados
    caminho_bd =  srbacktest['source'] 
    # Conectando ao banco
    conexao = sqlite3.connect(caminho_bd)
    # Lendo a view
    #consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
    consulta = f"""
    SELECT * FROM vwtitulosdados
    WHERE datetime BETWEEN '{srbacktest['dataini']}' AND '{srbacktest['datafim']}' AND symbol = '{srbacktest['symbol']}' AND intervalo = '{srbacktest['intervalo']}' AND moeda = '{srbacktest['moeda']}'
    ORDER BY datetime
    """
    dftitulosdados = pd.read_sql_query(consulta, conexao)
    # Fechando a conexão
    conexao.close()
    dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo"])
    return dftitulosdados

dftitulosdados = titulos_dados (srbacktest)
display(len(dftitulosdados))
display(dftitulosdados.head(10))

8957

,datetime,open,high,low,close,volume
0,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0
1,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0
2,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0
3,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0
4,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0
5,2020-04-06 13:00:00,5.8274,5.8354,5.7474,5.7834,68220.0
6,2020-04-06 14:00:00,5.7714,5.7794,5.6435,5.6435,84043.0
7,2020-04-06 15:00:00,5.6595,5.6595,5.5316,5.5956,122920.0
8,2020-04-06 16:00:00,5.6036,5.6036,5.6036,5.6036,49293.0
9,2020-04-07 09:00:00,5.9793,6.0672,5.7395,6.0032,101621.0


In [29]:
import sqlite3
import pandas as pd

                                #importa os parametros  e rangos dos scripts a executar
def scripts_parameters(id_backtest) :
    # Conecta ao banco de dados SQLite
    con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite
    
    # Consulta SQL para extrair o registro
    
    query = f"SELECT  name, type, max, min, step FROM vwbacktestparameters WHERE id_backtest = {id_backtest}"
    # Executa a consulta e lê em um DataFrame
    df = pd.read_sql_query(query, con)
    
    # Fecha a conexão (opcional)
    con.close()
    return df
dfcomb = scripts_parameters(id_backtest)
display (dfcomb)

,name,type,max,min,step
0,K,int,30,20,2
1,D,int,15,15,1
2,smoth,int,12,11,1
3,medM,int,6,6,1
4,lowM,int,5,5,1
5,stpl,float,0.02,0.02,0.01
6,comission,float,0.003,0.003,0.001
7,drawmax,float,0.2,0.02,0.1


In [31]:
                               

import pandas as pd
import numpy as np
from itertools import product

def parameters_combinator(dfcomb: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um DataFrame com todas as combinações possíveis de parâmetros
    definidos em dfcomb, respeitando os tipos especificados.

    Parâmetros esperados em dfcomb:
    - name: nome da coluna
    - type: tipo de dado ('int' ou 'float')
    - min: valor mínimo
    - max: valor máximo
    - step: incremento

    Retorna:
    - dfparamtest: DataFrame com todas as combinações possíveis
    """
    param_ranges = {}

    for _, row in dfcomb.iterrows():
        name = row['name']
        tipo = row['type']

        # Converte min, max, step para o tipo correto
        if tipo == 'int':
            min_val = int(row['min'])
            max_val = int(row['max'])
            step_val = int(row['step'])
        elif tipo == 'float':
            min_val = float(row['min'])
            max_val = float(row['max'])
            step_val = float(row['step'])
        else:
            raise ValueError(f"Tipo não suportado: {tipo}")

        # Gera a faixa de valores
        values = np.round(np.arange(min_val, max_val + step_val, step_val), 5)
        param_ranges[name] = values

    # Gera todas as combinações possíveis
    combinations = list(product(*param_ranges.values()))

    # Cria o novo DataFrame
    dfparamtest = pd.DataFrame(combinations, columns=param_ranges.keys())

    # Aplica os tipos definidos
    for _, row in dfcomb.iterrows():
        col = row['name']
        tipo = row['type']
        if tipo == 'int':
            dfparamtest[col] = dfparamtest[col].astype(int)
        elif tipo == 'float':
            dfparamtest[col] = dfparamtest[col].astype(float)

    return dfparamtest

dfparamtest = parameters_combinator(dfcomb)
print(dfparamtest.head())


    K   D  smoth  medM  lowM  stpl  comission  drawmax
0  20  15     11     6     5  0.02      0.003     0.02
1  20  15     11     6     5  0.02      0.003     0.12
2  20  15     11     6     5  0.02      0.003     0.22
3  20  15     12     6     5  0.02      0.003     0.02
4  20  15     12     6     5  0.02      0.003     0.12


In [33]:
print(len(dfparamtest)*len(dftitulosdados))

322452


In [64]:
dfmetricas = None

START LOOP

In [38]:

                       # loop de processamento de parametros verção  simple

def processar_parametros_simple (dfparamtest, il):
    
    for indice, linha in dfparamtest.iterrows():    
        il = indice
        
        K,D, smoth, medM, lowM, stpl, comission, drawmax, lsmetricas =   parametros (dfparamtest, il)
        
        dfsignals = Stoch_HighMedLow_Long (dftitulosdados, K, D, smoth, medM , lowM)
        
        dfsignals, dfstoploss = stop_loss_reentry (dfsignals, stpl)
       
        dfindex = index_calculation (dfsignals, comission)
        
        dfindexdrawdown = stop_drawdown_simple(dfindex, drawmax)
        
        # METRICS
        # creo dataframe para calculo de metricas
        dfinputmetricas = dfindex[['datetime', 'state','index_sc', 'index','trade']]
        
        setirtotalanual , lsmetricas = tir_total_anualizada (dfinputmetricas, lsmetricas)
        
        dftiranual = tir_anuais_df (dfinputmetricas, srbacktest['dataini'], srbacktest['datafim'])
        setiranuaisestats , lsmetricas = tir_anuais_estats(dftiranual, lsmetricas)
        
        setradesestats, lsmetricas = trades_estats(dfinputmetricas, lsmetricas)
        
        dfdrawdowns = drawdowns_df(dfinputmetricas)
        sedrawdownsestats , lsmetricas = drawdowns_estats(dfdrawdowns, lsmetricas)
       
        dfdiasout = dias_out_df (dfinputmetricas)
        sediasoutestats, lsmetricas = dias_out_estats(dfdiasout, lsmetricas)
       
        dfstopdrawdown = stop_drawdown_df (dfindexdrawdown)
        sestopdrawdownestats, lsmetricas = stop_drawdown_estats(dfstopdrawdown, lsmetricas)
        #display (lsmetricas )
        dfmetricas = atualizar_df_metricas(dfmetricas, lsmetricas)
        
        return dfmetricas
        
dfmetricas = processar_parametros_simple (dfparamtest, il)      


CPU times: total: 47 s
Wall time: 48.1 s


In [66]:
from concurrent.futures import ThreadPoolExecutor
import traceback

def processar_parametros_multinucleo(dfparamtest, dftitulosdados, srbacktest, dfmetricas):
    def processar_parametros(args):
        il, linha, dftitulosdados, dataini, datafim = args
        try:
            K, D, smoth, medM, lowM, stpl, comission, drawmax, lsmetricas = parametros(dfparamtest, il)

            dfsignals = Stoch_HighMedLow_Long(dftitulosdados, K, D, smoth, medM, lowM)
            dfsignals, dfstoploss = stop_loss_reentry(dfsignals, stpl)
            dfindex = index_calculation(dfsignals, comission)
            dfindexdrawdown = stop_drawdown_simple(dfindex, drawmax)

            dfinputmetricas = dfindex[['datetime', 'state', 'index_sc', 'index', 'trade']]
            setirtotalanual, lsmetricas = tir_total_anualizada(dfinputmetricas, lsmetricas)
            dftiranual = tir_anuais_df(dfinputmetricas, dataini, datafim)
            setiranuaisestats, lsmetricas = tir_anuais_estats(dftiranual, lsmetricas)
            setradesestats, lsmetricas = trades_estats(dfinputmetricas, lsmetricas)
            dfdrawdowns = drawdowns_df(dfinputmetricas)
            sedrawdownsestats, lsmetricas = drawdowns_estats(dfdrawdowns, lsmetricas)
            dfdiasout = dias_out_df(dfinputmetricas)
            sediasoutestats, lsmetricas = dias_out_estats(dfdiasout, lsmetricas)
            dfstopdrawdown = stop_drawdown_df(dfindexdrawdown)
            sestopdrawdownestats, lsmetricas = stop_drawdown_estats(dfstopdrawdown, lsmetricas)

            return lsmetricas

        except Exception as e:
            print(f"⚠️ Erro ao processar índice {il}: {e}")
            traceback.print_exc()
            return None

    # Prepara os argumentos para cada linha
    args_list = [
        (il, dfparamtest.iloc[il], dftitulosdados, srbacktest['dataini'], srbacktest['datafim'])
        for il in dfparamtest.index
    ]

    # Executa em paralelo
    resultados = []
    with ThreadPoolExecutor() as executor:
        for resultado in executor.map(processar_parametros, args_list):
            if resultado is not None:
                resultados.append(resultado)

    # Atualiza dfmetricas com os resultados válidos
    for lsmetricas in resultados:
        dfmetricas = atualizar_df_metricas(dfmetricas, lsmetricas)

    return dfmetricas

In [ ]:
dfmetricas = processar_parametros_multinucleo(dfparamtest, dftitulosdados, srbacktest, dfmetricas)

END LOOP

In [ ]:
dfmetricas.insert(
    loc=0,  # insere como primeira coluna
    column="id_backtest",
    value=[srbacktest["id_backtest"]] * len(dfmetricas)
)
display(dfmetricas)